<a href="https://colab.research.google.com/github/tougheye/Data_processing/blob/main/accreted_title_step_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The goal of the project is to take in TCS backend data then for each title -  of

1.   get the min, mid, max
2.   count the number of steps in the scale

This project is to support the step calculation of accreted dtitles

In [1]:
import pandas as pd

tcs_backend_folder = "/content/drive/MyDrive/Data/UCOP_Job_Codes_Summary"

In [2]:
tcs_backend_file = pd.ExcelFile(f'{tcs_backend_folder}/R-349 Job Codes Summary_07282026.xlsx')
tcs_backend_file.sheet_names
#

['Job Code Review',
 'Represented Job Codes',
 'Non-Represented Job Codes',
 'Shift and On Call Rates']

In [3]:
tcs_backend_tabs = tcs_backend_file.sheet_names
represented_job_codes_df = tcs_backend_file.parse('Represented Job Codes', skiprows=9, skipfooter=2)
represented_job_codes_df.shape

(128803, 30)

In [4]:
represented_active_df = represented_job_codes_df[represented_job_codes_df['Salary Grade Eff Status'] == 'A']
represented_active_df.shape

(109454, 30)

In [5]:
upte_represented_job_codes_df = represented_active_df[represented_active_df['Union Code'].isin(['HX', 'RX', 'TX'])]
upte_represented_job_codes_df.shape

(40459, 30)

In [6]:
upte_job_title_Eff_date_cnt = upte_represented_job_codes_df.groupby(['Salary Plan SETID','Job Code Description'])['Eff Date - Salary Grade'].nunique().reset_index(name='Date Count')
multi_eff_date_setid_jobCode_cnt = upte_job_title_Eff_date_cnt.where(upte_job_title_Eff_date_cnt['Date Count'] > 1).dropna()

In [7]:
# get the latest effective date for each job title
latest_eff_date_df = upte_represented_job_codes_df.groupby(['Salary Plan SETID','Job Code Description'])['Eff Date - Salary Grade'].max().reset_index(name='Latest Effective Date')

In [8]:
# filtered UPTE represented job codes
titles_not_updated = upte_represented_job_codes_df[upte_represented_job_codes_df['Eff Date - Salary Grade'] < '2026-07-01']
titles_not_updated.shape   # 1192 rows


(1192, 30)

In [9]:
# Merge to get the latest effective date for each job title for which the effective date is before July 1, 2026
# WILL KEEP THIS ONE TO LATER TAKE CARE OF
titles_not_updated_w_max_date = titles_not_updated.merge(latest_eff_date_df, on=['Salary Plan SETID', 'Job Code Description'], how='inner')

# Drop the LBNL business units as they are not supposed to receive the 5% ATB in July 2026
titles_not_updated_w_max_date = titles_not_updated_w_max_date[titles_not_updated_w_max_date['Salary Plan SETID'] != 'LBNL1']
titles_not_updated_w_max_date.shape       # 1130 rows

(1130, 31)

In [10]:

# Filter the rows with the latest effective date for each job title that have multiple effective dates
# THIS DF WILL BE CONCATENATED LATER TO CREATE THE FINAL DF WITH THE LATEST EFFECTIVE DATE

upte_represented_max_eff_date_df = upte_represented_job_codes_df\
    .merge(multi_eff_date_setid_jobCode_cnt,
           on=['Salary Plan SETID', 'Job Code Description'],
           how='inner')\
    .sort_values('Eff Date - Salary Grade', ascending=False)\
    .drop_duplicates(['Salary Plan SETID', 'Job Code Description', 'UCPATH Step', 'UC  Half Step'],
                     keep='first')

upte_represented_max_eff_date_df.shape

(1566, 31)

In [11]:
# Create unique identifier both in the master UPTE represented df and multiple effective date df

represented_key = upte_represented_job_codes_df['Salary Plan SETID'] + ' - ' + upte_represented_job_codes_df['Union Code'] + ' - ' + upte_represented_job_codes_df['Job Code Description']
unmatched_key = upte_represented_max_eff_date_df['Salary Plan SETID'] + ' - ' + upte_represented_max_eff_date_df['Union Code'] + ' - ' + upte_represented_max_eff_date_df['Job Code Description']

# filtering out the multiple effective date rows from the UPTE represented df based on the unique identifier keys
upte_represented_one_date_df = upte_represented_job_codes_df[~represented_key.isin(unmatched_key)]

In [12]:
upte_represented_final_df = pd.concat([upte_represented_one_date_df, upte_represented_max_eff_date_df.drop('Date Count', axis=1)])
upte_represented_final_df.shape

(39392, 30)

In [13]:
cols_to_keep = ['Salary Plan SETID', 'Job Code', 'Job Code Description', 'Union Code', 'UCPATH Step', 'UC  Half Step', 'Hrly Rate',
                'Eff Date - Salary Grade', 'Salary Grade Eff Status']
#

In [14]:
upte_represented_job_codes_refined_df = upte_represented_final_df[cols_to_keep]


In [17]:
upte_represented_job_codes_refined_df.sample(10)

,Salary Plan SETID,Job Code,Job Code Description,Union Code,UCPATH Step,UC Half Step,Hrly Rate,Eff Date - Salary Grade,Salary Grade Eff Status
27663,IRCMP,009393,PSYCHOMETRIST,HX,6.0,6.0,34.370000,2026-07-05,A
28148,IRCMP,009613,SRA 1,RX,1.0,1.0,29.990000,2026-07-05,A
4434,BKCMP,009520,SPECTROSCOPIST,RX,5.0,5.0,46.872342,2026-07-01,A
18306,DVMED,008653,LAB MECHN,TX,3.0,2.0,37.100000,2026-07-05,A
58851,MECMP,009610,SRA 4,RX,13.0,13.0,49.906221,2026-07-01,A
104169,SFMED,007954,RECR THER 2 EX,HX,12.0,12.0,57.780000,2026-07-05,A
104916,SFMED,008811,NUC MED TCHNO PRN,HX,6.0,6.0,100.040000,2026-07-05,A
2558,BKCMP,007102,DRAFTING TCHN SR,TX,7.0,4.0,32.690000,2026-07-05,A
76631,SDCMP,005160,USER EXP DESIGNER 2 TX,TX,NaN,NaN,NaN,2026-07-01,A
75653,SDCMP,004804,COMPUTER RESC SPEC 2,TX,2.0,1.5,36.020000,2026-07-05,A


Codes above are all from the [TCS payscale notebook](https://colab.research.google.com/drive/1MwFYF32tmy4eGs_cS6p0rc_hfDQ3drTM?usp=chrome_ntp#scrollTo=H9473xBkUotq)

In [20]:
import math
import numpy as np
business_unit = 'BKCMP'

job_details_dict = {}
  #define the current business unit
current_bus_unit = upte_represented_job_codes_refined_df[
      upte_represented_job_codes_refined_df['Salary Plan SETID'] == business_unit]

# loop through the bargaining units
for bargaining_unit in current_bus_unit['Union Code'].unique():
  job_details_dict[bargaining_unit] = {}

  # define the current bargaining unit
  current_bus_unit_bu_df = current_bus_unit[current_bus_unit['Union Code'] == bargaining_unit]

  title_list = sorted(current_bus_unit_bu_df['Job Code Description'].unique())

  for title in title_list:
    current_title_df = current_bus_unit_bu_df[current_bus_unit_bu_df['Job Code Description'] == title]

    # Filter out NaN values from 'UC Half Step' to get valid steps
    valid_half_steps = [float(k) for k in current_title_df['UC  Half Step'].unique() if not math.isnan(k)]

    half_step_list = sorted(valid_half_steps)

    # Initialize Mid Step and Mid Step Rate to NaN
    mid_step_index = len(half_step_list) // 2
    if mid_step_index == 0:
      continue

    mid_step_value = half_step_list[mid_step_index]

    total_steps = len(half_step_list)

    # Find the row in current_title_df that matches this mid_step_value
    mid_step_rate_series = current_title_df[current_title_df['UC  Half Step'] == mid_step_value]['Hrly Rate']

    if not mid_step_rate_series.empty:
        mid_step_rate = mid_step_rate_series.iloc[0]

    job_details_dict[bargaining_unit][title] = {
          'Title' : title,
          'Job Code' : current_title_df['Job Code'].unique()[0],
          'Min Rate' : current_title_df['Hrly Rate'].min(),
          'Max Rate' : current_title_df['Hrly Rate'].max(),
          'Total Steps' : total_steps,
          'TCS Effective Date': current_bus_unit_bu_df[current_bus_unit_bu_df['Job Code Description'] == title]['Eff Date - Salary Grade'].unique().strftime("%m/%d/%Y"),
          'Mid Step': mid_step_value,
          'Mid Step Rate': mid_step_rate
    }

In [26]:
import math
# dictionary to store the job details
job_details_dict = {}

business_unit_list = upte_represented_job_codes_refined_df['Salary Plan SETID'].unique()

for business_unit in business_unit_list:
  job_details_dict[business_unit] = {}
  #define the current business unit
  current_bus_unit = upte_represented_job_codes_refined_df[
      upte_represented_job_codes_refined_df['Salary Plan SETID'] == business_unit]

  # loop through the bargaining units
  for bargaining_unit in current_bus_unit['Union Code'].unique():
    job_details_dict[business_unit][bargaining_unit] = {}

    # define the current bargaining unit
    current_bus_unit_bu_df = current_bus_unit[current_bus_unit['Union Code'] == bargaining_unit]

    title_list = sorted(current_bus_unit_bu_df['Job Code Description'].unique())

    for title in title_list:
      current_title_df = current_bus_unit_bu_df[current_bus_unit_bu_df['Job Code Description'] == title]

      # Filter out NaN values from 'UC Half Step' to get valid steps
      valid_half_steps = [float(k) for k in current_title_df['UC  Half Step'].unique() if not math.isnan(k)]

      half_step_list = sorted(valid_half_steps)

      # Initialize Mid Step and Mid Step Rate to NaN
      mid_step_index = len(half_step_list) // 2
      if mid_step_index == 0:
        continue

      mid_step_value = half_step_list[mid_step_index]

      total_steps = len(half_step_list)

      # Find the row in current_title_df that matches this mid_step_value
      mid_step_rate_series = current_title_df[current_title_df['UC  Half Step'] == mid_step_value]['Hrly Rate']

      if not mid_step_rate_series.empty:
          mid_step_rate = mid_step_rate_series.iloc[0]

      job_details_dict[business_unit][bargaining_unit][title] = {
            'Title' : title,
            'Job Code' : current_title_df['Job Code'].unique()[0],
            'Min Rate' : current_title_df['Hrly Rate'].min(),
            'Max Rate' : current_title_df['Hrly Rate'].max(),
            'Total Steps' : total_steps,
            'TCS Effective Date': current_bus_unit_bu_df[current_bus_unit_bu_df['Job Code Description'] == title]['Eff Date - Salary Grade'].unique().strftime("%m/%d/%Y"),
            'Mid Step': mid_step_value,
            'Mid Step Rate': mid_step_rate
      }



In [42]:
col_order = ['Title', 'Job Code', 'Min Rate','Mid Step Rate','Max Rate',
             'Total Steps','Mid Step','TCS Effective Date']

all_dfs = {}
for business_unit, bargaining_units in job_details_dict.items():
  for bargaining_unit, titles in bargaining_units.items(): # Corrected to iterate over items to get the 'titles' dictionary
    df_name = f"{business_unit}_{bargaining_unit}_df"

    if titles: # Check if the titles dictionary is not empty
        df = pd.DataFrame(titles).T

        # Drop 'Title' column if it exists, as the index will become the Title column after reset_index
        if 'Title' in df.columns:
            df = df.drop('Title', axis=1)

        df.index.name = 'Title'
        df.reset_index(inplace=True)

        # Ensure all columns from col_order are present, filling with pd.NA if missing
        for col in col_order:
            if col not in df.columns:
                df[col] = pd.NA

        all_dfs[df_name] = df[col_order] # Reorder and select columns
        print(f"Created dataframe {df_name} with columns {all_dfs[df_name].columns}")
    else:
        # If 'titles' is empty, create an empty DataFrame with the expected columns
        all_dfs[df_name] = pd.DataFrame(columns=col_order)
        print(f"Created empty dataframe {df_name} (no titles found) with columns {all_dfs[df_name].columns}")

Created dataframe BKCMP_TX_df with columns Index(['Title', 'Job Code', 'Min Rate', 'Mid Step Rate', 'Max Rate',
       'Total Steps', 'Mid Step', 'TCS Effective Date'],
      dtype='object')
Created dataframe BKCMP_HX_df with columns Index(['Title', 'Job Code', 'Min Rate', 'Mid Step Rate', 'Max Rate',
       'Total Steps', 'Mid Step', 'TCS Effective Date'],
      dtype='object')
Created dataframe BKCMP_RX_df with columns Index(['Title', 'Job Code', 'Min Rate', 'Mid Step Rate', 'Max Rate',
       'Total Steps', 'Mid Step', 'TCS Effective Date'],
      dtype='object')
Created dataframe DVCMP_TX_df with columns Index(['Title', 'Job Code', 'Min Rate', 'Mid Step Rate', 'Max Rate',
       'Total Steps', 'Mid Step', 'TCS Effective Date'],
      dtype='object')
Created dataframe DVCMP_HX_df with columns Index(['Title', 'Job Code', 'Min Rate', 'Mid Step Rate', 'Max Rate',
       'Total Steps', 'Mid Step', 'TCS Effective Date'],
      dtype='object')
Created dataframe DVCMP_RX_df with columns In

In [44]:
all_dfs['BKCMP_HX_df'].head()

,Title,Job Code,Min Rate,Mid Step Rate,Max Rate,Total Steps,Mid Step,TCS Effective Date
0,BEH HEALTH COUNSELOR 2 HX,004458,54.62,61.51,67.9,12,7.0,[07/05/2026]
1,BEH HEALTH COUNSELOR 3 HX,004459,60.093807,67.676489,74.723673,12,7.0,"[07/01/2026, 12/19/2023]"
2,BEH HEALTH COUNSELOR 4 HX,004460,66.102122,74.445211,82.188539,12,7.0,[07/01/2026]
3,CLIN LAB SCI,008940,59.1,70.65,82.78,18,10.0,[07/05/2026]
4,CLIN LAB SCI SPEC,008939,65.04,77.73,91.09,18,10.0,[07/05/2026]


In [ ]:
writer
for key in all_dfs.keys():
